# ForecastLLM - Week 6 Day 2: Feature Engineering for Time Series

This week - build a model that predicts future values from historical data.

A model that can estimate the next value in a time series, from recent observations.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Baselines and Evaluation  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll transform a cleaned series into supervised features.
Lag-based features are a strong and simple starting point.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Feature Engineering</h2>
            <span style="color:#181;">Feature engineering translates raw sequential data into model-ready inputs.
            For many forecasting problems, simple lag features provide a fast baseline and often strong practical value.</span>
        </td>
    </tr>
</table>


In [1]:
import numpy as np
import pandas as pd

# replaced pricing/text pipeline with local time-series loading
from week6.data_loader import load_sample_series


# The next cell is where you choose data source

`USE_FALLBACK = True` keeps this lab runnable even before full M4 wiring.

For this lab, we start with one primary series and engineer forecasting features.


In [2]:
USE_FALLBACK = True

In [3]:
# Load one sample series (fallback to synthetic if needed)

def make_synthetic_series(periods=220):
    dates = pd.date_range("2024-01-01", periods=periods, freq="D")
    trend = np.linspace(50, 70, periods)
    weekly = 4 * np.sin(np.arange(periods) * 2 * np.pi / 7)
    noise = np.random.default_rng(42).normal(scale=1.2, size=periods)
    return pd.DataFrame({"timestamp": dates, "value": trend + weekly + noise})

loaded = None
try:
    loaded = load_sample_series()
except Exception as e:
    print(f"Loader failed ({e}); using fallback")

if isinstance(loaded, pd.Series):
    ts_df = loaded.rename("value").to_frame().reset_index()
    if ts_df.shape[1] == 2:
        ts_df.columns = ["timestamp", "value"]
elif isinstance(loaded, pd.DataFrame):
    ts_df = loaded.copy()
else:
    ts_df = make_synthetic_series()

if "value" not in ts_df.columns:
    numeric_cols = ts_df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        ts_df = ts_df.rename(columns={numeric_cols[0]: "value"})
    else:
        ts_df = make_synthetic_series()

if "timestamp" not in ts_df.columns:
    ts_df = ts_df.reset_index().rename(columns={ts_df.index.name or "index": "timestamp"})

ts_df = ts_df[["timestamp", "value"]].copy()
ts_df["value"] = pd.to_numeric(ts_df["value"], errors="coerce")
ts_df = ts_df.dropna(subset=["value"]).copy()
ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], errors="coerce")
if ts_df["timestamp"].isna().all() and USE_FALLBACK:
    ts_df = make_synthetic_series()

ts_df = ts_df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {len(ts_df):,} observations")
print(ts_df.head(2))


Loaded 220 observations
   timestamp      value
0 2024-01-01  50.365660
1 2024-01-02  51.970669


In [4]:
ts_df.head(5)

,timestamp,value
0,2024-01-01,50.365660
1,2024-01-02,51.970669
2,2024-01-03,54.982901
3,2024-01-04,53.138185
4,2024-01-05,46.288520


In [5]:
# Give every row an id for easier inspection

ts_df["row_id"] = np.arange(len(ts_df))
ts_df[["row_id", "timestamp", "value"]].head(3)


,row_id,timestamp,value
0,0,2024-01-01,50.365660
1,1,2024-01-02,51.970669
2,2,2024-01-03,54.982901


In [6]:
# Lag feature configuration
LAG_COLUMNS = ["lag_1", "lag_2", "lag_3", "lag_7"]

In [7]:
print(ts_df[["timestamp", "value"]].head(10).to_string(index=False))

 timestamp     value
2024-01-01 50.365660
2024-01-02 51.970669
2024-01-03 54.982901
2024-01-04 53.138185
2024-01-05 46.288520
2024-01-06 44.994294
2024-01-07 47.574028
2024-01-08 50.259778
2024-01-09 53.837758
2024-01-10 53.697977


In [8]:
# Build lag features (replaced embedding/features workflow)

df = ts_df[["timestamp", "value"]].copy()
df["lag_1"] = df["value"].shift(1)
df["lag_2"] = df["value"].shift(2)
df["lag_3"] = df["value"].shift(3)
df["lag_7"] = df["value"].shift(7)

# target is the value at the current row, using prior rows as predictors
df["target"] = df["value"]

df.head(10)


,timestamp,value,lag_1,lag_2,lag_3,lag_7,target
0,2024-01-01,50.365660,NaN,NaN,NaN,NaN,50.365660
1,2024-01-02,51.970669,50.365660,NaN,NaN,NaN,51.970669
2,2024-01-03,54.982901,51.970669,50.365660,NaN,NaN,54.982901
3,2024-01-04,53.138185,54.982901,51.970669,50.365660,NaN,53.138185
4,2024-01-05,46.288520,53.138185,54.982901,51.970669,NaN,46.288520
5,2024-01-06,44.994294,46.288520,53.138185,54.982901,NaN,44.994294
6,2024-01-07,47.574028,44.994294,46.288520,53.138185,NaN,47.574028
7,2024-01-08,50.259778,47.574028,44.994294,46.288520,50.365660,50.259778
8,2024-01-09,53.837758,50.259778,47.574028,44.994294,51.970669,53.837758
9,2024-01-10,53.697977,53.837758,50.259778,47.574028,54.982901,53.697977


In [9]:
# Inspect NaNs introduced by lagging
df.isna().sum()

timestamp    0
value        0
lag_1        1
lag_2        2
lag_3        3
lag_7        7
target       0
dtype: int64

In [10]:
FEATURE_COLUMNS = LAG_COLUMNS
TARGET_COLUMN = "target"

In [11]:
def make_supervised(frame, feature_columns, target_column="target"):
    out = frame.copy()
    out = out.dropna(subset=feature_columns + [target_column]).copy()
    return out


In [12]:
df.head(1)

,timestamp,value,lag_1,lag_2,lag_3,lag_7,target
0,2024-01-01,50.36566,NaN,NaN,NaN,NaN,50.36566


In [13]:
make_supervised(df, FEATURE_COLUMNS, TARGET_COLUMN).head()

,timestamp,value,lag_1,lag_2,lag_3,lag_7,target
7,2024-01-08,50.259778,47.574028,44.994294,46.288520,50.365660,50.259778
8,2024-01-09,53.837758,50.259778,47.574028,44.994294,51.970669,53.837758
9,2024-01-10,53.697977,53.837758,50.259778,47.574028,54.982901,53.697977
10,2024-01-11,53.704055,53.697977,53.837758,50.259778,53.138185,53.704055
11,2024-01-12,50.202382,53.704055,53.697977,53.837758,46.288520,50.202382


In [14]:
def chronological_split(frame, train_frac=0.8):
    cutoff_idx = int(len(frame) * train_frac)
    train = frame.iloc[:cutoff_idx].copy()
    test = frame.iloc[cutoff_idx:].copy()
    return train, test


In [15]:
supervised_df = make_supervised(df, FEATURE_COLUMNS, TARGET_COLUMN)
supervised_df.shape

(213, 7)

In [16]:
# Optional calendar features from timestamp (kept simple)
supervised_df["day_of_week"] = supervised_df["timestamp"].dt.dayofweek
supervised_df["month"] = supervised_df["timestamp"].dt.month


In [17]:
# Feature matrix and target vector
MODEL_FEATURES = FEATURE_COLUMNS + ["day_of_week", "month"]
X = supervised_df[MODEL_FEATURES].copy()
y = supervised_df[TARGET_COLUMN].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (213, 6)
y shape: (213,)


In [18]:
# Chronological split (no shuffle)
split_idx = int(len(supervised_df) * 0.8)

X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

print("Split index:", split_idx)


Split index: 170


In [19]:
(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

((170, 6), (43, 6), (170,), (43,))

In [20]:
# Quick safety checks before modeling
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert X_train.index.max() < X_test.index.min()
print("Chronological split checks passed")


Chronological split checks passed


In [21]:
print(supervised_df[["timestamp"] + MODEL_FEATURES + [TARGET_COLUMN]].head(10).to_string(index=False))

 timestamp     lag_1     lag_2     lag_3     lag_7  day_of_week  month    target
2024-01-08 47.574028 44.994294 46.288520 50.365660            0      1 50.259778
2024-01-09 50.259778 47.574028 44.994294 51.970669            1      1 53.837758
2024-01-10 53.837758 50.259778 47.574028 54.982901            2      1 53.697977
2024-01-11 53.697977 53.837758 50.259778 53.138185            3      1 53.704055
2024-01-12 53.704055 53.697977 53.837758 46.288520            4      1 50.202382
2024-01-13 50.202382 53.704055 53.697977 44.994294            5      1 47.275416
2024-01-14 47.275416 50.202382 53.704055 47.574028            6      1 49.412578
2024-01-15 49.412578 47.275416 50.202382 50.259778            0      1 51.839550
2024-01-16 51.839550 49.412578 47.275416 53.837758            1      1 53.466038
2024-01-17 53.466038 51.839550 49.412578 53.697977            2      1 55.803400


In [22]:
print(X_train.head(5))

        lag_1      lag_2      lag_3      lag_7  day_of_week  month
7   47.574028  44.994294  46.288520  50.365660            0      1
8   50.259778  47.574028  44.994294  51.970669            1      1
9   53.837758  50.259778  47.574028  54.982901            2      1
10  53.697977  53.837758  50.259778  53.138185            3      1
11  53.704055  53.697977  53.837758  46.288520            4      1


In [23]:
print(y_train.head(5))

7     50.259778
8     53.837758
9     53.697977
10    53.704055
11    50.202382
Name: target, dtype: float64


## I've put exactly this logic into helper functions

- Builds lag features from one cleaned time series
- Drops lag-related NaNs
- Creates chronological train/test split without shuffling

This gives us a reliable Day 2 baseline for forecasting experiments.


In [ ]:
# earlier duplicate helper execution removed; keeping the later version that adds calendar features

In [24]:
# run helper-based pipeline end-to-end
supervised_df = make_supervised(df, FEATURE_COLUMNS, TARGET_COLUMN)
supervised_df["day_of_week"] = supervised_df["timestamp"].dt.dayofweek
supervised_df["month"] = supervised_df["timestamp"].dt.month
train_df, test_df = chronological_split(supervised_df, train_frac=0.8)

print(f"Supervised rows: {len(supervised_df):,}")
print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")


Supervised rows: 213
Train rows: 170
Test rows: 43


In [25]:
# final sanity check for missing values
print("NaNs in X_train:", int(X_train.isna().sum().sum()))
print("NaNs in X_test:", int(X_test.isna().sum().sum()))


NaNs in X_train: 0
NaNs in X_test: 0


In [26]:
# inspect the last prepared row
supervised_df.tail(1)

,timestamp,value,lag_1,lag_2,lag_3,lag_7,target,day_of_week,month
219,2024-08-07,73.064581,72.850595,71.422229,67.24444,73.2972,73.064581,2,8


In [27]:
# keep ordering explicit for forecasting (no shuffle)
print("Train time range:", train_df["timestamp"].min(), "->", train_df["timestamp"].max())
print("Test time range:", test_df["timestamp"].min(), "->", test_df["timestamp"].max())


Train time range: 2024-01-08 00:00:00 -> 2024-06-25 00:00:00
Test time range: 2024-06-26 00:00:00 -> 2024-08-07 00:00:00


In [28]:
# inspect one training example
idx = X_train.index[0]
print("Features:")
print(X_train.loc[idx])
print("Target:", y_train.loc[idx])


Features:
lag_1          47.574028
lag_2          44.994294
lag_3          46.288520
lag_7          50.365660
day_of_week     0.000000
month           1.000000
Name: 7, dtype: float64
Target: 50.259778295580396


In [29]:
# remove fields that we don't need for immediate model input
model_ready_train = train_df[MODEL_FEATURES + [TARGET_COLUMN]].copy()
model_ready_test = test_df[MODEL_FEATURES + [TARGET_COLUMN]].copy()

model_ready_train.head(2)


,lag_1,lag_2,lag_3,lag_7,day_of_week,month,target
7,47.574028,44.994294,46.288520,50.365660,0,1,50.259778
8,50.259778,47.574028,44.994294,51.970669,1,1,53.837758


## Persist step (deferred)

In the original notebook, Day 2 pushes pre-processed data artifacts externally.

For ForecastLLM Day 2, we keep artifacts local and move directly to Day 3 baselines.


In [30]:
# local artifacts for Day 3
artifacts = {
    "features": MODEL_FEATURES,
    "target": TARGET_COLUMN,
    "split": {"train_rows": len(model_ready_train), "test_rows": len(model_ready_test)},
}

print(artifacts)


{'features': ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'day_of_week', 'month'], 'target': 'target', 'split': {'train_rows': 170, 'test_rows': 43}}


## Prepared outputs

- `X_train`, `X_test`, `y_train`, `y_test`
- `model_ready_train`, `model_ready_test`

These are ready for baseline models and MAE/sMAPE evaluation in Day 3.
